# Lista 2 - Regressão logística e métodos estatísticos

## Questão 1

Considere o conjunto de dados disponível em **breastcancer.csv**, organizado em 31 colunas, sendo as 30 primeiras colunas os atributos e a última coluna a saída. Os 30 atributos coletados de exames médicos são usados no diagnóstico do câncer de mama, sendo 1 a classe positiva e 0 a classe negativa.

### a) Considerando uma validação cruzada em 10 *folds*, avalie modelos de classificação binária nos dados em questão. Para tanto, use as abordagens abaixo:

* **Regressão logística** (treinado com GD ou SGD);
* **Análise do discriminante Gaussiano**;
* **Naive Bayes Gaussiano**;

### **Solução:**

In [4]:
import sys
sys.path.append("../")

#### **Regressão Logística via GD**

In [1]:
# 1. Importação dos pacotes necessários
from utils.utils import *
from models.regressaologistica import RegressaoLogistica, grid_search_RL
from preprocessors.normalizador import Normalizador

In [2]:
# 2. Extração e divisão treino-teste dos dados
y = breastcancer[:,[-1]]
X = breastcancer[:, :-1]

X_train, X_test, y_train, y_test = treine_teste_divida(X, y)

In [3]:
# 3. Normalização dos dados de treinamento via MinMaxScaler
normalizadorRL = Normalizador(X_train)
X_train_normalizado = normalizadorRL.normaliza(X_train)

In [4]:
# 4. Otimização dos hiperparâmetros via grid search com k-fold cross validation (com k = 10)
Xy_train_normalizado = np.hstack([X_train_normalizado, y_train.reshape(-1,1)])

# Definindo o espaço de busca dos hiperparêmtros da Regressão Logística
n = 10 # Número de candidatos
hyperparameters_search_space_RL = {
    "alpha": list(np.linspace(0.0001, 0.1, num=n)) + [0.01],
    "lambda": list(np.linspace(0.001, 500, num=n)) + [0]
}

# Realizando o processo de Grid Search onde é salvo a melhor acurácia e o melhor conjunto de hiperparâmetros
best_accuracy_RL, best_hyperparameters_RL = grid_search_RL(hyperparameters_search_space_RL, Xy_train_normalizado, 10)
print(f"Melhores hiperparâmetros obtidos: (alpha, lambda) = {best_hyperparameters_RL}\nMelhor acurácia média obtida: {best_accuracy_RL}")

Melhores hiperparâmetros obtidos: (alpha, lambda) = (0.044500000000000005, 0.001)
Melhor acurácia média obtida: 0.9676688251618872


#### **Análise de Discriminante Gaussiano**

In [6]:
# 1. Importação dos pacotes necessários
from utils import *
from models.analise_de_discriminante_gaussiano import Analise_de_Discriminante_Gaussiano, grid_search_ADG
from preprocessors.normalizador import Normalizador

In [7]:
# 2. Extração e divisão treino-teste dos dados
y = breastcancer[:,[-1]]
X = breastcancer[:, :-1]

X_train, X_test, y_train, y_test = treine_teste_divida(X, y)

In [8]:
# 3. Normalização dos dados de treinamento
normalizadorADG = Normalizador(X_train)
X_train_normalizado = normalizadorADG.normaliza(X_train)

In [9]:
# 4. Otimização de hiperparâmetros via grid seach com k-fold cross validation (com k = 10)
Xy_train_normalizado = np.hstack([X_train_normalizado, y_train.reshape(-1,1)])

# Definindo o espaço de busca de como a priori é computada no modelo
hyperparameters_search_space_ADG = {
    "priori": ["equiprovavel","relativa"]
}

# Realizando o processo de Grid Search onde é salvo a melhor acurácia e o melhor conjunto de hiperparâmetros
best_accuracy_ADG, best_hyperparameters_ADG = grid_search_ADG(hyperparameters_search_space_ADG, Xy_train_normalizado, 10, 2)
print(f"Melhores hiperparâmetros obtidos: (priori) = {best_hyperparameters_ADG}\nMelhor acurácia média obtida: {best_accuracy_ADG}")

Melhores hiperparâmetros obtidos: (priori) = ('equiprovavel',)
Melhor acurácia média obtida: 0.5417755256251063


#### **Naive Bayes Gaussiano**

In [11]:
# 1. Importação dos pacotes necessários
from utils.utils import *
from models.naive_bayes_gaussiano import Naive_Bayes_Gaussiano, grid_search_NBG
from preprocessors.normalizador import Normalizador

In [12]:
# 2. Extração e divisão treino-teste dos dados
y = breastcancer[:,[-1]]
X = breastcancer[:, :-1]

X_train, X_test, y_train, y_test = treine_teste_divida(X, y)

In [13]:
# 3. Normalização dos dados de treinamento
normalizadorNBG = Normalizador(X_train)
X_train_normalizado = normalizadorNBG.normaliza(X_train)

In [14]:
# 4 Otimização de hiperparâmetros
Xy_train_normalizado = np.hstack([X_train_normalizado, y_train.reshape(-1,1)])

# Definindo o espaço de busca de como a priori é computada no modelo
hyperparameters_search_space_NBG = {
    "priori": ["equiprovavel","relativa"]
}

# Realizando o processo de Grid Search onde é salvo a melhor acurácia e o melhor conjunto de hiperparâmetros
best_accuracy_NBG, best_hyperparameters_NBG = grid_search_NBG(hyperparameters_search_space_NBG, Xy_train_normalizado, 10, 2)
print(f"Melhores hiperparâmetros obtidos: (priori) = {best_hyperparameters_NBG}\nMelhor acurácia média obtida: {best_accuracy_NBG}")

Melhores hiperparâmetros obtidos: (priori) = ('relativa',)
Melhor acurácia média obtida: 0.5453901422347658


### b) Para cada modelo criado, reporte valor médio e desvio padrão da **acurácia global** e da **acurácia por classe**.

### **Solução**

#### **Regressão Logística via GD**

In [13]:
# 5. Treinamento do modelo

# Inicializando o classificador com os hiperparâmetros já otimizados
classificador_binario_RL = RegressaoLogistica(*best_hyperparameters_RL)

# Treinando o modelo
classificador_binario_RL.ajuste(X_train_normalizado, y_train)

In [14]:
# 6. Teste do modelo (avaliação)

# Normalizando os dados de entradas com as mesmas estatísticas da normalização de treino
X_test_normalizado = normalizadorRL.normaliza(X_test)

# Realizando a classificação de novos valores com esses dados de teste normalizados
y_pred = classificador_binario_RL.prever(X_test_normalizado)

# Obtendo as acurácias do modelo
acuracia_global_media, acuracia_global_desvio = acc(y_test, y_pred.reshape(-1,1), 2)
acuracias_medias, acuracias_desvios = acc(y_test, y_pred.reshape(-1,1), 2, classe="local")


# Resultados
print("===== Resultados da Regressão Logística =====")
print(f"Acurácia global (valor médio): {acuracia_global_media}")
print(f"Acurácia global (desvio padrão): {acuracia_global_desvio}")
for i in range(2):
    print(f"\nAcurácia da classe {i} (valor médio): {acuracias_medias[i]}")
    print(f"Acurácia da classe {i} (desvio padrão): {acuracias_desvios[i]}")

===== Resultados da Regressão Logística =====
Acurácia global (valor médio): 0.956140350877193
Acurácia global (desvio padrão): 0.20478276368296058

Acurácia da classe 0 (valor médio): 1.0
Acurácia da classe 0 (desvio padrão): 0.0

Acurácia da classe 1 (valor médio): 0.8837209302325582
Acurácia da classe 1 (desvio padrão): 0.32055927330442374


#### **Análise de Discriminante Gaussiano**

In [15]:
# 5. Treinamento do modelo

# Inicializando o classificador com os hiperparâmetros já otimizados
classificador_binario_ADG = Analise_de_Discriminante_Gaussiano(2, *best_hyperparameters_ADG)

# Treinando o modelo
classificador_binario_ADG.ajuste(X_train_normalizado, y_train)

In [16]:
# 6. Teste do modelo

# Normalizando os dados de entradas com as mesmas estatísticas da normalização de treino
X_test_normalizado = normalizadorADG.normaliza(X_test)

# Realizando a classificação de novos valores com esses dados de teste normalizados
y_pred = classificador_binario_ADG.prever(X_test_normalizado)

# Obtendo as acurácias do modelo
acuracia_global_media, acuracia_global_desvio = acc(y_test, y_pred.reshape(-1,1))
acuracias_medias, acuracias_desvios = acc(y_test, y_pred.reshape(-1,1), 2, classe="local")

# Resultados
print("===== Resultados da Análise de Discriminante Gaussiano =====")
print(f"Acurácia global (valor médio): {acuracia_global_media}")
print(f"Acurácia global (desvio padrão): {acuracia_global_desvio}")
for i in range(2):
    print(f"\nAcurácia da classe {i} (valor médio): {acuracias_medias[i]}")
    print(f"Acurácia da classe {i} (desvio padrão): {acuracias_desvios[i]}")

===== Resultados da Análise de Discriminante Gaussiano =====
Acurácia global (valor médio): 0.9298245614035088
Acurácia global (desvio padrão): 0.25544245225545675

Acurácia da classe 0 (valor médio): 0.9577464788732394
Acurácia da classe 0 (desvio padrão): 0.2011669979871225

Acurácia da classe 1 (valor médio): 0.8837209302325582
Acurácia da classe 1 (desvio padrão): 0.32055927330442374


#### **Naive Bayes Gaussiano**

In [17]:
# 5. Treinamento do modelo

# Inicializando o classificador com os hiperparâmetros já otimizados
classificador_binario_NBG = Naive_Bayes_Gaussiano(2, *best_hyperparameters_NBG)

# Treinando o modelo
classificador_binario_NBG.ajuste(X_train_normalizado, y_train)

In [18]:
# 6. Teste do modelo

# Normalizando os dados de entradas com as mesmas estatísticas da normalização de treino
X_test_normalizado = normalizadorNBG.normaliza(X_test)

# Realizando a classificação de novos valores com esses dados de teste normalizados
y_pred = classificador_binario_NBG.prever(X_test_normalizado)

# Obtendo as acurácias do modelo
acuracia_global_media, acuracia_global_desvio = acc(y_test, y_pred.reshape(-1,1))
acuracias_medias, acuracias_desvios = acc(y_test, y_pred.reshape(-1,1), 2, classe="local")

# Resultados
print("===== Resultados do Naive Bayes Gaussiano =====")
print(f"Acurácia global (valor médio): {acuracia_global_media}")
print(f"Acurácia global (desvio padrão): {acuracia_global_desvio}")
for i in range(2):
    print(f"\nAcurácia da classe {i} (valor médio): {acuracias_medias[i]}")
    print(f"Acurácia da classe {i} (desvio padrão): {acuracias_desvios[i]}")

===== Resultados do Naive Bayes Gaussiano =====
Acurácia global (valor médio): 0.9122807017543859
Acurácia global (desvio padrão): 0.2828862367824053

Acurácia da classe 0 (valor médio): 0.9154929577464789
Acurácia da classe 0 (desvio padrão): 0.2781467275793169

Acurácia da classe 1 (valor médio): 0.9069767441860465
Acurácia da classe 1 (desvio padrão): 0.29046502318132084


## Questão 2

Considere o conjunto de dados disponível em **vehicle.csv**, organizado em 19 colunas, sendo as 18 primeiras colunas os atributos e a última coluna a saída. Os 18 atrubutos caracterizam a silhueta de veículos, extraídos pelo método HIPS (Hierarchical Image Processing System). A tarefa consiste em classificar o veículo em 4 classes (bus, opel, saab, e van).

### a) Considerando uma validação cruzada em 10 *folds*, avalie modelos de classificação multiclasse nos dados em questão. Para tanto, use as abordagens abaixo:
* **Regressão softmax** (treinado com GD ou SGD);
* **Análise do discriminante Gaussiano**;
* **Naive Bayes Gaussiano**;

### Solução

#### **Regressão softmax via GD**

In [20]:
# 1. Importação dos pacotes necessários
from utils.utils import *
from models.regressaosoftmax import RegressaoSoftmax, grid_search_RS
from preprocessors.normalizador import Normalizador

In [21]:
# 2. Extração dos dados
y = vehicle[:,[-1]]
X = vehicle[:, :-1]

# 3. Codificação da variável target via One-Hot-Encoding
y = one_hot_encoding(y,4)

# 4. divisão dos dados nos conjuntos de treino e teste
X_train, X_test, y_train, y_test = treine_teste_divida(X, y)

In [22]:
# 5. Normalização dos dados
normalizadorRS = Normalizador(X_train)
X_train_normalizado = normalizadorRS.normaliza(X_train)

In [24]:
# 6. Otimização de hiperparâmetros
Xy_train_normalizado = np.hstack([X_train_normalizado, y_train])

# Definindo o espaço de busca dos hiperparâmetros da regressão softmax
n = 2 # Número de candidatos
hyperparameters_search_space_RS = {
    "alpha": list(np.linspace(0.0001, 0.1, num=n)) + [0.01],
    "lambda": list(np.linspace(0.001, 500, num=n)) + [0]
}

# Realizando o processo de Grid Search onde é salvo a melhor acurácia e o melhor conjunto de hiperparâmetros
best_accuracy_RS2, best_hyperparameters_RS2 = grid_search_RS(hyperparameters_search_space_RS, Xy_train_normalizado, 10, 4)
print(f"Melhores hiperparâmetros obtidos: (alpha, lambda) = {best_hyperparameters_RS2}\nMelhor acurácia média obtida: {best_accuracy_RS2}")

Melhores hiperparâmetros obtidos: (alpha, lambda) = (0.1, 0)
Melhor acurácia média obtida: 0.25266054334925714


#### **Análise do discriminante Gaussiano**

In [23]:
# 1. Importação dos pacotes necessários
from utils.utils import *
from models.analise_de_discriminante_gaussiano import Analise_de_Discriminante_Gaussiano, grid_search_ADG
from preprocessors.normalizador import Normalizador

In [24]:
# 2. Extração dos dados
y = vehicle[:,[-1]]
X = vehicle[:, :-1]

# 3. Codificação da variável target via One-Hot-Encoding
y = one_hot_encoding(y,4)

# 4. divisão dos dados nos conjuntos de treino e teste
X_train, X_test, y_train, y_test = treine_teste_divida(X, y)

In [25]:
# 5. Normalização dos dados
normalizadorADG = Normalizador(X_train)
X_train_normalizado = normalizadorADG.normaliza(X_train)

In [26]:
# 6. Otimização de hiperparâmetros
Xy_train_normalizado = np.hstack([X_train_normalizado, y_train])

# Definindo o espaço de busca de como a priori é computada no modelo
hyperparameters_search_space_ADG = {
    "priori": ["equiprovavel","relativa"]
}

# Realizando o processo de Grid Search onde é salvo a melhor acurácia e o melhor conjunto de hiperparâmetros
best_accuracy_ADG, best_hyperparameters_ADG = grid_search_ADG(hyperparameters_search_space_ADG, Xy_train_normalizado, 10, 4)
print(f"Melhores hiperparâmetros obtidos: (priori) = {best_hyperparameters_ADG}\nMelhor acurácia média obtida: {best_accuracy_ADG}")

Melhores hiperparâmetros obtidos: (priori) = ('relativa',)
Melhor acurácia média obtida: 0.8509803921568627


#### **Naive Bayes Gaussiano**

In [27]:
# 1. Importação dos pacotes necessários
from utils.utils import *
from models.naive_bayes_gaussiano import Naive_Bayes_Gaussiano, grid_search_NBG
from preprocessors.normalizador import Normalizador

In [28]:
# 2. Extração dos dados
y = vehicle[:,[-1]]
X = vehicle[:, :-1]

# 3. Codificação da variável target via One-Hot-Encoding
y = one_hot_encoding(y,4)

# 4. divisão dos dados nos conjuntos de treino e teste
X_train, X_test, y_train, y_test = treine_teste_divida(X, y)

In [29]:
# 5. Normalização dos dados de treinamento
normalizadorNBG = Normalizador(X_train)
X_train_normalizado = normalizadorNBG.normaliza(X_train)

In [30]:
# 6. Otimização de hiperparâmetros
Xy_train_normalizado = np.hstack([X_train_normalizado, y_train])

# Definindo o espaço de busca de como a priori é computada no modelo
hyperparameters_search_space_NBG = {
    "priori": ["equiprovavel","relativa"]
}

# Realizando o processo de Grid Search onde é salvo a melhor acurácia e o melhor conjunto de hiperparâmetros
best_accuracy_NBG, best_hyperparameters_NBG = grid_search_NBG(hyperparameters_search_space_NBG, Xy_train_normalizado, 10, 4)
print(f"Melhores hiperparâmetros obtidos: (priori) = {best_hyperparameters_NBG}\nMelhor acurácia média obtida: {best_accuracy_NBG}")

Melhores hiperparâmetros obtidos: (priori) = ('relativa',)
Melhor acurácia média obtida: 0.44974424552429665


### b) Para cada modelo criado, reporte valor médio e desvio padrão da **acurácia global** e da **acurácia por classe**.

### Solução

#### **Regressão softmax via GD**

In [33]:
# 7. Treinamento do modelo

# Inicializando o classificador com os hiperparâmetros já otimizados
classificador_multiclasse_RS = RegressaoSoftmax(*best_hyperparameters_RS2)

# Treinando o modelo
classificador_multiclasse_RS.ajuste(X_train_normalizado, y_train)

In [34]:
# 8. Teste do modelo (avaliação)

# Normalizando os dados de entradas com as mesmas estatísticas da normalização de treino
X_test_normalizado = normalizadorRS.normaliza(X_test)

# Realizando a classificação de novos valores com esses dados de teste normalizados
y_pred = classificador_multiclasse_RS.prever(X_test_normalizado)

# Obtendo as acurácias do modelo
acuracia_global_media, acuracia_global_desvio = acc(np.argmax(y_test, axis=1).reshape(-1,1), y_pred, 4)
acuracias_medias, acuracias_desvios = acc(np.argmax(y_test, axis=1).reshape(-1,1), y_pred, 4, classe="local")


# Resultados
print("===== Resultados da Regressão Softmax =====")
print(f"Acurácia global (valor médio): {acuracia_global_media}")
print(f"Acurácia global (desvio padrão): {acuracia_global_desvio}")
for i in range(4):
    print(f"\nAcurácia da classe {i} (valor médio): {acuracias_medias[i]}")
    print(f"Acurácia da classe {i} (desvio padrão): {acuracias_desvios[i]}")

===== Resultados da Regressão Softmax =====
Acurácia global (valor médio): 0.7529411764705882
Acurácia global (desvio padrão): 0.4313012418781967

Acurácia da classe 0 (valor médio): 0.4444444444444444
Acurácia da classe 0 (desvio padrão): 0.49690399499995336

Acurácia da classe 1 (valor médio): 0.625
Acurácia da classe 1 (desvio padrão): 0.4841229182759271

Acurácia da classe 2 (valor médio): 1.0
Acurácia da classe 2 (desvio padrão): 0.0

Acurácia da classe 3 (valor médio): 0.9555555555555556
Acurácia da classe 3 (desvio padrão): 0.20608041101101565


#### **Análise do discriminante Gaussiano**

In [31]:
# 7. Treinamento do modelo

# Inicializando o classificador com os hiperparâmetros já otimizados
classificador_multiclasse_ADG = Analise_de_Discriminante_Gaussiano(4, *best_hyperparameters_ADG)

# Treinando o modelo
classificador_multiclasse_ADG.ajuste(X_train_normalizado, y_train)

In [32]:
# 8. Teste do modelo (avaliação)

# Normalizando os dados de entradas com as mesmas estatísticas da normalização de treino
X_test_normalizado = normalizadorADG.normaliza(X_test)

# Realizando a classificação de novos valores com esses dados de teste normalizados
y_pred = classificador_multiclasse_ADG.prever(X_test_normalizado)

# Obtendo as acurácias do modelo
acuracia_global_media, acuracia_global_desvio = acc(np.argmax(y_test, axis=1).reshape(-1,1), y_pred.reshape(-1,1), 4)
acuracias_medias, acuracias_desvios = acc(np.argmax(y_test, axis=1).reshape(-1,1), y_pred.reshape(-1,1), 4, classe="local")

# Resultados
print("===== Resultados da Análise de Discriminante Gaussiano =====")
print(f"Acurácia global (valor médio): {acuracia_global_media}")
print(f"Acurácia global (desvio padrão): {acuracia_global_desvio}")
for i in range(4):
    print(f"\nAcurácia da classe {i} (valor médio): {acuracias_medias[i]}")
    print(f"Acurácia da classe {i} (desvio padrão): {acuracias_desvios[i]}")

===== Resultados da Análise de Discriminante Gaussiano =====
Acurácia global (valor médio): 0.8823529411764706
Acurácia global (desvio padrão): 0.32218973970892123

Acurácia da classe 0 (valor médio): 0.7948717948717948
Acurácia da classe 0 (desvio padrão): 0.4037952755903493

Acurácia da classe 1 (valor médio): 0.7142857142857143
Acurácia da classe 1 (desvio padrão): 0.45175395145262554

Acurácia da classe 2 (valor médio): 1.0
Acurácia da classe 2 (desvio padrão): 0.0

Acurácia da classe 3 (valor médio): 1.0
Acurácia da classe 3 (desvio padrão): 0.0


#### **Naive Bayes Gaussiano**

In [33]:
# 7. Treinamento do modelo

# Inicializando o classificador com os hiperparâmetros já otimizados
classificador_multiclasse_NBG = Naive_Bayes_Gaussiano(4, *best_hyperparameters_NBG)

# Treinando o modelo
classificador_multiclasse_NBG.ajuste(X_train_normalizado, y_train)

In [34]:
# 8. Teste do modelo (avaliação)

# Normalizando os dados de entradas com as mesmas estatísticas da normalização de treino
X_test_normalizado = normalizadorNBG.normaliza(X_test)

# Realizando a classificação de novos valores com esses dados de teste normalizados
y_pred = classificador_multiclasse_NBG.prever(X_test_normalizado)

# Obtendo as acurácias do modelo
acuracia_global_media, acuracia_global_desvio = acc(np.argmax(y_test, axis=1).reshape(-1,1), y_pred.reshape(-1,1), 4)
acuracias_medias, acuracias_desvios = acc(np.argmax(y_test, axis=1).reshape(-1,1), y_pred.reshape(-1,1), 4, classe="local")

# Resultados
print("===== Resultados do Naive Bayes Gaussiano =====")
print(f"Acurácia global (valor médio): {acuracia_global_media}")
print(f"Acurácia global (desvio padrão): {acuracia_global_desvio}")
for i in range(4):
    print(f"\nAcurácia da classe {i} (valor médio): {acuracias_medias[i]}")
    print(f"Acurácia da classe {i} (desvio padrão): {acuracias_desvios[i]}")

===== Resultados do Naive Bayes Gaussiano =====
Acurácia global (valor médio): 0.49411764705882355
Acurácia global (desvio padrão): 0.4999653967264888

Acurácia da classe 0 (valor médio): 0.4358974358974359
Acurácia da classe 0 (desvio padrão): 0.4958738360465055

Acurácia da classe 1 (valor médio): 0.5
Acurácia da classe 1 (desvio padrão): 0.5

Acurácia da classe 2 (valor médio): 0.9111111111111111
Acurácia da classe 2 (desvio padrão): 0.28458329944145994

Acurácia da classe 3 (valor médio): 0.11363636363636363
Acurácia da classe 3 (desvio padrão): 0.31736909190383955
